### Structured Data
When claude needs to generate structured data like JSON, python code or bulleted lists, a problem with claude generation shows up, claude wants to be helpful and will add explanatory text and comments around the code

- The solution is to use Assistant Message Prefilling + Stop Sequences:
    - user message tells Claude what to generate
    - prefilled assistant message makes Claude think it already started a markdown code block
    - Claude continues by writing just the JSON content
    - When Claude tries to close the code block with ```, the stop sequence immediately ends generation
- General Formula For Structured Data:
    1. check what the default prefilling messages looks like
    2. find the repeating starting and ending prefilling message
    3. add the starting prefilling to *Assistant Message Prefilling*
    4. add the ending prefilling to *Stop Sequences*

Example: Consider building a web app that generates AWS EventBridge rules
- Users enter a description, click generate, and expect to see clean JSON to immediately copy and use
- If Claude returns the JSON wrapped in markdown code blocks with explanatory text and comments
- users can't copy the entire response, they will have to select only the JSON portion


In [9]:
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [11]:
def chat(messages, system=None, temperature=1.0, stop_sequences=None): # adding in the new parameter into the chat function
    params = {
        "model": model,
        "max_tokens": 250,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    
    if stop_sequences: 
        # is defined in the same way as system: default being None
        # and adding it in the way of an if statement to check if one is there or not, adding it to the params set if it is
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)
    return message.content[0].text

def add_user_message(messages, text): 
    user_message = {"role": "user", "content": text} 
    messages.append(user_message) 

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

In [12]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

answer = chat(messages)
answer

# this default response contains: ```json ... ``` for formatting into markdown text
# however this still contains markdown formatting and explanatory text, this is not practical for applications where users only need the raw code

'```json\n{\n  "Name": "MySimpleRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n```'

In [14]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json") # this makes claude think that it has already generated this, so it skips the markdown initiation going straight to the json generation

text = chat(messages, stop_sequences=["```"]) # after claude is done with json it will want to close the markdown it thinks it started
# the stop sequence tells claude that when it is about to generate ``` the generation needs to come to an end by giving claude a Stop event, skipping the closing of the markdown generation

text
# returns only the json, with only \n that can be removed using .strip() function

'\n{\n  "Name": "MyRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'

In [15]:
import json

json.loads(text.strip())

{'Name': 'MyRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}

### Structured Data Exercise

- using message prefilling and stop sequences to get 3 different commands in a single response
- without any comments or explanations
- message prefilling doesn't only consist of characters like: ```

In [35]:
messages = []

prompt = """
Generate three different AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
text = chat(messages)

text

'# Three AWS CLI Commands\n\n1. **List all S3 buckets:**\n```bash\naws s3 ls\n```\n\n2. **Describe EC2 instances:**\n```bash\naws ec2 describe-instances\n```\n\n3. **Get current AWS account ID:**\n```bash\naws sts get-caller-identity\n```'

In [18]:
from IPython.display import Markdown

Markdown(text)

# Three AWS CLI Commands

1. **List all S3 buckets:**
```bash
aws s3 ls
```

2. **Get current AWS account ID:**
```bash
aws sts get-caller-identity
```

3. **List EC2 instances in default region:**
```bash
aws ec2 describe-instances
```

#### Personal Attempts

In [25]:
# notes:
    # multiple stop sequences can be added in the stop_sequences list
    # not too sure on how to add more than 1 in message prefilling, since thats only at the beginning of the message
    # could use multiple prefilling and chat prompts
    # key take away: what ever you start off with in message prefilling is what the model believes it has already written and will continue to generate from that starting point, so it is possible to dictate the output of the model

messages = []

In [28]:
# attempt 1: getting rid of the first main header
messages = []

add_user_message(messages, prompt)
add_assistant_message(messages, "# Three AWS CLI Commands")
text = chat(messages)
Markdown(text)

# the main header is gone, but the individual headers for each of the three points still remain
# problem with this is that theres code that is required between each individual header, so the new line character after each point could get claude to stop generating early



1. **List all S3 buckets:**
```bash
aws s3 ls
```

2. **Get current AWS account ID:**
```bash
aws sts get-caller-identity
```

3. **Describe all EC2 instances:**
```bash
aws ec2 describe-instances
```

In [31]:
messages = []

# attempt 2: getting rid of the points headers
    # which start with:
        # **List all S3 buckets:**
        # \n```
    # and end with:
        # ```\n
    # the points are separated and can be identified by \n1, \n2, \n3
# the main problem still is adding more than 1 in message prefilling and having code generate in between
# idea: using the chat function more than once
    # stopping the code generation, then continuing with the code generation with more messages and add_assistant_message (including more message prefilling)

add_user_message(messages, prompt)
add_assistant_message(messages, "# Three AWS CLI Commands") # after the heading is generated, a /n will be generated, then the headers for the individual points
# after this the first heading will appear
add_assistant_message(messages, "\n1 **List all S3 buckets:**")

text1 = chat(messages, stop_sequences=["\n2"]) # for the second header

Markdown(text1)
# this results in only the first command appearing with nothing else


```bash
aws s3 ls
```


In [34]:
messages

[{'role': 'user',
  'content': '\nGenerate three different AWS CLI commands. Each should be very short.\n'},
 {'role': 'assistant', 'content': '# Three AWS CLI Commands'},
 {'role': 'assistant', 'content': '\n1 **List all S3 buckets:**'},
 {'role': 'user',
  'content': '\nGenerate three different AWS CLI commands. Each should be very short.\n'},
 {'role': 'assistant', 'content': ' \n2 **Get current AWS account ID:**'}]

In [33]:
# attempt 3: adding back the text back into add user message 

add_user_message(messages, prompt) # this includes: # Three AWS CLI Commands, \n1 **List all S3 buckets:** 
# the next thing needed to generate is the second header: **Get current AWS account ID:**
add_assistant_message(messages, " \n2 **Get current AWS account ID:**")
text2 = chat(messages, stop_sequences=["\n3"]) # for the third header

Markdown(text2) # this only includes the second output


```bash
aws sts get-caller-identity --query Account --output text
```


#### Solution
- what threw me off was my generation was different from the generation in the video (my generation jumped straight into a main header instead of the code block)
- The concept that I didn't grasp fully was just how powerful message prefilling really is, if I want it to start off with markdown code generation, I just need to add the starting text for it
    - it is also possible to inject prompt engineering into message prefilling, by starting the generation and setting down ground rules like no comments and no headers 

In [38]:
messages = []

prompt = """
Generate three different AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "```")
text = chat(messages, stop_sequences=["```"])

text.strip()
# this method still has `bash` and will occasionally still have comments included in the results

'bash\n# 1. List all S3 buckets\naws s3 ls\n\n# 2. Describe EC2 instances\naws ec2 describe-instances\n\n# 3. Get IAM user info\naws iam get-user'

In [41]:
messages = []

add_user_message(messages, prompt)
add_assistant_message(messages, "```bash")
text = chat(messages, stop_sequences=["```"])

text.strip()
# this method removes `bash`, but will only give one function at a time

'aws ec2 describe-instances'

In [43]:
messages = []

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three different commands in a single block without any comments: \n ```bash")
text = chat(messages, stop_sequences=["```"])

text.strip()

'aws s3 ls\naws ec2 describe-instances\naws lambda list-functions'